In [4]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

load_dotenv(Path(project_root) / ".env")

ARVAN_AI_API_KEY = os.getenv("ARVAN_AI_API_KEY")
ARVAN_EMBEDDING_URL = os.getenv("ARVAN_EMBEDDING_URL")

print("Project root:", project_root)
print("API key loaded:", bool(ARVAN_AI_API_KEY))
print("Gateway URL loaded:", bool(ARVAN_EMBEDDING_URL))
print("Gateway URL preview:", ARVAN_EMBEDDING_URL[:60] + "..." if ARVAN_EMBEDDING_URL else None)

Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance
API key loaded: True
Gateway URL loaded: True
Gateway URL preview: https://arvancloudai.ir/gateway/models/Embedding-3-Large/rAg...


In [5]:
from openai import OpenAI

client = OpenAI(
    api_key=ARVAN_AI_API_KEY,
    base_url=ARVAN_EMBEDDING_URL
)

response = client.embeddings.create(
    model="text-embedding-3-large",
    input="این یک تست کوتاه برای امبدینگ فارسی است."
)

embedding = response.data[0].embedding

print("Embedding created successfully")
print("Embedding length:", len(embedding))
print("First 5 values:", embedding[:5])

Embedding created successfully
Embedding length: 3072
First 5 values: [-0.041595458984375, 0.03900146484375, 0.0005116462707519531, 0.0013704299926757812, 0.05267333984375]


#### سلول های بالا تست بود
سلول ۳ — خواندن chunks.json

In [6]:
import json
from pathlib import Path

CHUNKS_FILE = Path(project_root) / "data" / "chunks" / "komeil_roudi" / "chunks.json"
EMBEDDINGS_DIR = Path(project_root) / "data" / "embeddings" / "komeil_roudi"

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Chunks file:", CHUNKS_FILE)
print("Embeddings directory:", EMBEDDINGS_DIR)
print("Loaded chunks:", len(chunks))

print("\nFirst chunk ID:", chunks[0]["chunk_id"])
print("First chunk text length:", len(chunks[0]["text"]))
print("First chunk preview:")
print(chunks[0]["text"][:500])

Chunks file: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/chunks/komeil_roudi/chunks.json
Embeddings directory: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/embeddings/komeil_roudi
Loaded chunks: 168

First chunk ID: 0A050EE9_1
First chunk text length: 1132
First chunk preview:
هر فیلمی به موضوعی می‌پردازد و پرداختن به آن را در قالب گره اصلی و فرعی و اشارات می‌گنجاند. کارگردان به ویژه در طول فیلم فریاد می‌زند که «من پیامی با پرداختن به این موضوع دارم». گاه زبان فیلم نارساست و گاه گوش با واژه‌ها ناآشنا؛ اما سینما زمانی سینماست که پیامی رسا به واژه‌هایی آشنا بیان شود. این هر دو به ترتیب به این معناست که اول، فیلم بیان سینمایی دارد و دوم، مخاطب‌فهم و مخاطب‌پسند است. زمانی که فیلم بیان سینمایی ندارد، از سینما فاصله می‌گیرد؛ هر چند در سینما به نمایش درآید. سینما هنر هفتم و 


سلول ۴ — تست Embedding روی اولین Chunk

In [7]:
test_chunk = chunks[0]

response = client.embeddings.create(
    model="text-embedding-3-large",
    input=test_chunk["text"]
)

test_embedding = response.data[0].embedding

print("Chunk ID:", test_chunk["chunk_id"])
print("Text length:", len(test_chunk["text"]))
print("Embedding created successfully")
print("Embedding length:", len(test_embedding))
print("First 5 values:", test_embedding[:5])

Chunk ID: 0A050EE9_1
Text length: 1132
Embedding created successfully
Embedding length: 3072
First 5 values: [-0.0260467529296875, 0.0550537109375, -0.02117919921875, 0.0021610260009765625, 0.04559326171875]


سلول ۵ — ساخت تابع Embedding برای یک متن

In [9]:
def get_embedding(text):
    """
    Create embedding for a single text using Arvan AI Gateway.
    """

    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )

    return response.data[0].embedding

سلول ۶ — ساخت Embedding برای همه Chunkها و ذخیره خروجی

In [10]:
import time
from tqdm import tqdm

embedded_chunks = []

for chunk in tqdm(chunks):
    embedding = get_embedding(chunk["text"])

    embedded_chunk = {
        "chunk_id": chunk["chunk_id"],
        "article_id": chunk["article_id"],
        "chunk_index": chunk["chunk_index"],
        "title": chunk["title"],
        "url": chunk["url"],
        "text": chunk["text"],
        "char_count": chunk["char_count"],
        "embedding": embedding,
        "embedding_model": "text-embedding-3-large"
    }

    embedded_chunks.append(embedded_chunk)

    # Small delay to avoid rate limit
    time.sleep(0.2)

output_file = EMBEDDINGS_DIR / "embeddings.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(embedded_chunks, f, ensure_ascii=False, indent=2)

print("Total embedded chunks:", len(embedded_chunks))
print("Saved to:", output_file)
print("Embedding length:", len(embedded_chunks[0]["embedding"]))

100%|██████████| 168/168 [04:57<00:00,  1.77s/it]


Total embedded chunks: 168
Saved to: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/embeddings/komeil_roudi/embeddings.json
Embedding length: 3072


سلول ۷ — بررسی فایل embeddings.json

In [11]:
with open(EMBEDDINGS_DIR / "embeddings.json", "r", encoding="utf-8") as f:
    embedded_chunks_check = json.load(f)

print("Loaded embedded chunks:", len(embedded_chunks_check))

first_item = embedded_chunks_check[0]

print("\nKeys:")
print(first_item.keys())

print("\nChunk ID:", first_item["chunk_id"])
print("Title:", first_item["title"])
print("Text length:", len(first_item["text"]))
print("Embedding length:", len(first_item["embedding"]))
print("Embedding model:", first_item["embedding_model"])
print("First 5 embedding values:", first_item["embedding"][:5])

Loaded embedded chunks: 168

Keys:
dict_keys(['chunk_id', 'article_id', 'chunk_index', 'title', 'url', 'text', 'char_count', 'embedding', 'embedding_model'])

Chunk ID: 0A050EE9_1
Title: سه میزان اصلی در ارزیابی فیلم‌های سینمایی از منظر سواد مالی
Text length: 1132
Embedding length: 3072
Embedding model: text-embedding-3-large
First 5 embedding values: [-0.0260467529296875, 0.0550537109375, -0.02117919921875, 0.0021610260009765625, 0.04559326171875]


<div style="direction: rtl; white-space: normal; line-height: 1;">
عالی. ✅
مرحله **Embeddings** کامل و سالم تمام شد.

وضعیت فعلی پروژه:

```text
✅ Ingestion
✅ Preprocessing
✅ Chunking
✅ Embeddings with Arvan
⬅️ Vector Store
⬜ Retrieval
⬜ Generation
```

خروجی نهایی این مرحله:

```text
data/embeddings/komeil_roudi/embeddings.json
```

این فایل شامل:

```text
168 embedded chunks
dimension: 3072
model: text-embedding-3-large
```

مرحله بعدی: **Vector Store**.

در مرحله بعد باید این embeddingها را داخل یک ساختار قابل جستجو بریزیم. با توجه به تنظیمات پروژه‌ات، مسیر طبیعی بعدی **Qdrant** است.

</div>